In [2]:

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Load the CSV file
file_path = Path("Assignment-1_Data.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "Please place Assignment-1_Data.csv in the same folder as your notebook."
    )

df = pd.read_csv(
    file_path,
    sep=";",
    encoding="latin1",
    low_memory=False
)

# Fix the column names
df.columns = (
    df.columns
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())


Dataset loaded successfully!
Dataset shape: (522064, 7)
Columns: ['BillNo', 'Itemname', 'Quantity', 'Date', 'Price', 'CustomerID', 'Country']


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,"2,55",17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,"2,75",17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,"3,39",17850.0,United Kingdom


In [3]:

# Check the dataset size
print("Total records:", len(df))
print("Total transactions:", df["BillNo"].nunique())
print("Total countries:", df["Country"].nunique())
print("Missing product names:", df["Itemname"].isna().sum())

# Count records and transactions for each country
country_summary = (
    df.groupby("Country")
    .agg(
        Records=("BillNo", "size"),
        Transactions=("BillNo", "nunique")
    )
    .sort_values("Transactions", ascending=False)
)

display(country_summary)


Total records: 522064
Total transactions: 21663
Total countries: 30
Missing product names: 1455


,Records,Transactions
Country,,
United Kingdom,487622,20122
Germany,9042,457
France,8408,392
Belgium,2031,98
Netherlands,2363,95
Spain,2485,90
Portugal,1501,58
Australia,1185,57
Switzerland,1967,54


In [4]:

# Make a copy of the original dataset
clean_df = df.copy()

# Clean column values
clean_df["BillNo"] = clean_df["BillNo"].astype(str).str.strip()
clean_df["Itemname"] = (
    clean_df["Itemname"]
    .astype("string")
    .str.strip()
    .str.upper()
)
clean_df["Quantity"] = pd.to_numeric(
    clean_df["Quantity"], errors="coerce"
)
clean_df["Price"] = pd.to_numeric(
    clean_df["Price"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

# Identify possible cancellation bills
cancellation_mask = clean_df["BillNo"].str.upper().str.startswith("C")

print("Original rows:", len(clean_df))
print("Missing product names:", clean_df["Itemname"].isna().sum())
print("Non-positive quantities:", (clean_df["Quantity"] <= 0).sum())
print("Possible cancellation bills:", cancellation_mask.sum())

# Keep valid product purchases
clean_df = clean_df[
    clean_df["BillNo"].ne("")
    & clean_df["Itemname"].notna()
    & clean_df["Itemname"].ne("")
    & (clean_df["Quantity"] > 0)
    & ~cancellation_mask
].copy()

# Remove extra spaces from product names
clean_df["Itemname"] = clean_df["Itemname"].str.replace(
    r"\s+", " ", regex=True
)

print("\nRows after cleaning:", len(clean_df))
print("Transactions after cleaning:", clean_df["BillNo"].nunique())
print("Unique products:", clean_df["Itemname"].nunique())
print("Countries remaining:", clean_df["Country"].nunique())

display(clean_df.head())


Original rows: 522064
Missing product names: 1455
Non-positive quantities: 1336
Possible cancellation bills: 0

Rows after cleaning: 520136
Transactions after cleaning: 19735
Unique products: 4043
Countries remaining: 30


,BillNo,Itemname,Quantity,Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01.12.2010 08:26,2.55,17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01.12.2010 08:26,3.39,17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01.12.2010 08:26,2.75,17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01.12.2010 08:26,3.39,17850.0,United Kingdom
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,01.12.2010 08:26,3.39,17850.0,United Kingdom


In [5]:

# Group products by transaction
transactions = (
    clean_df.groupby("BillNo")["Itemname"]
    .apply(lambda items: sorted(set(items)))
)

print("Total transaction baskets:", len(transactions))
print("\nFirst 5 transaction baskets:")

for bill_no, items in transactions.head().items():
    print(f"\nBillNo: {bill_no}")
    print("Products:", items)


Total transaction baskets: 19735

First 5 transaction baskets:

BillNo: 536365
Products: ['CREAM CUPID HEARTS COAT HANGER', 'GLASS STAR FROSTED T-LIGHT HOLDER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN']

BillNo: 536366
Products: ['HAND WARMER RED POLKA DOT', 'HAND WARMER UNION JACK']

BillNo: 536367
Products: ['ASSORTED COLOUR BIRD ORNAMENT', 'BOX OF 6 ASSORTED COLOUR TEASPOONS', 'BOX OF VINTAGE ALPHABET BLOCKS', 'BOX OF VINTAGE JIGSAW BLOCKS', 'DOORMAT NEW ENGLAND', 'FELTCRAFT PRINCESS CHARLOTTE DOLL', 'HOME BUILDING BLOCK WORD', 'IVORY KNITTED MUG COSY', 'LOVE BUILDING BLOCK WORD', "POPPY'S PLAYHOUSE BEDROOM", "POPPY'S PLAYHOUSE KITCHEN", 'RECIPE BOX WITH METAL HEART']

BillNo: 536368
Products: ['BLUE COAT RACK PARIS FASHION', 'JAM MAKING SET WITH JARS', 'RED COAT RACK PARIS FASHION', 'YELLOW COAT RACK PARIS FASHION']

BillNo: 536369
Products: ['BATH BUILDING BL

In [6]:

from mlxtend.preprocessing import TransactionEncoder

# Convert the transaction baskets into a one-hot encoded table
te = TransactionEncoder()

encoded_array = te.fit(transactions).transform(
    transactions,
    sparse=True
)

basket_sets = pd.DataFrame.sparse.from_spmatrix(
    encoded_array,
    index=transactions.index,
    columns=te.columns_
)

print("Basket table shape:", basket_sets.shape)
print("\nFirst 5 transactions:")
display(basket_sets.head())


ModuleNotFoundError: No module named 'mlxtend'

In [ ]:
%pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import sys

print("Python executable:", sys.executable)

import mlxtend
print("mlxtend version:", mlxtend.__version__)

Python executable: c:\Users\bea\anaconda3\python.exe
mlxtend version: 0.25.0


In [ ]:

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

encoded_array = te.fit(transactions).transform(
    transactions,
    sparse=True
)

basket_sets = pd.DataFrame.sparse.from_spmatrix(
    encoded_array,
    index=transactions.index,
    columns=te.columns_
)

print("Basket table shape:", basket_sets.shape)
display(basket_sets.head())

Basket table shape: (19735, 4043)


C:\Users\bea\AppData\Local\Temp\ipykernel_23560\955255649.py:10: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_sets = pd.DataFrame.sparse.from_spmatrix(


,*BOOMBOX IPOD CLASSIC,*USB OFFICE MIRROR BALL,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
BillNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:

from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

encoded_array = te.fit(transactions).transform(
    transactions,
    sparse=True
)

basket_sets = pd.DataFrame.sparse.from_spmatrix(
    encoded_array,
    index=transactions.index,
    columns=te.columns_
)

print("Basket table shape:", basket_sets.shape)
display(basket_sets.head())

C:\Users\bea\AppData\Local\Temp\ipykernel_23560\955255649.py:10: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_sets = pd.DataFrame.sparse.from_spmatrix(


Basket table shape: (19735, 4043)


,*BOOMBOX IPOD CLASSIC,*USB OFFICE MIRROR BALL,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
BillNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
frequent_itemsets = apriori(
    basket_sets,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    "support",
    ascending=False
)

print("Number of frequent itemsets:", len(frequent_itemsets))
display(frequent_itemsets.head(20))

NameError: name 'apriori' is not defined

In [ ]:

# Generate association rules
if not frequent_itemsets.empty and frequent_itemsets["itemsets"].map(len).max() >= 2:

    rules = association_rules(
        frequent_itemsets,
        metric="lift",
        min_threshold=1
    )

    rules = rules.sort_values(
        ["lift", "confidence"],
        ascending=False
    )

    print("Total association rules:", len(rules))

    # Display the most important columns
    display(
        rules[
            ["antecedents", "consequents", "support", "confidence", "lift"]
        ].head(20)
    )

else:
    rules = pd.DataFrame()
    print("No frequent product combinations found.")
    print("The support threshold may need to be lowered.")

NameError: name 'frequent_itemsets' is not defined

In [ ]:

if not rules.empty:

    # Filter strong association rules
    strong_rules = rules[
        (rules["lift"] >= 8) &
        (rules["confidence"] >= 0.8)
    ].sort_values(
        ["lift", "confidence"],
        ascending=False
    )

    print("Total strong rules:", len(strong_rules))
    display(
        strong_rules[
            ["antecedents", "consequents", "support", "confidence", "lift"]
        ]
    )

    # Plot support vs confidence
    plt.figure(figsize=(10, 6))
    plt.scatter(
        rules["support"],
        rules["confidence"],
        alpha=0.6
    )

    plt.xlabel("Support")
    plt.ylabel("Confidence")
    plt.title("Association Rules - All Countries")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

else:
    strong_rules = pd.DataFrame()
    print("No association rules available to filter or plot.")


NameError: name 'rules' is not defined